# 🌸 SweetGF - Google Colab Setup

This notebook sets up SweetGF on Google Colab with Ollama and ComfyUI.

**Note:** This will use Colab's GPU resources. Make sure you have GPU runtime enabled.

## Setup Steps:
1. Run each cell in order
2. Wait for installations to complete
3. Access the app through the provided ngrok URL

## 1. Install System Dependencies

In [ ]:
# Install required system packages
!apt-get update -qq
!apt-get install -y -qq wget git python3-pip

## 2. Clone SweetGF Repository

In [ ]:
import os

# Clone your repository (replace with your actual repo URL)
if not os.path.exists('/content/SweetGF'):
    # Option 1: Clone from GitHub (if you have it there)
    # !git clone https://github.com/yourusername/SweetGF.git /content/SweetGF
    
    # Option 2: Upload files manually
    print("Please upload your SweetGF files to /content/SweetGF")
    print("Or uncomment the git clone command above if you have a GitHub repo")
    !mkdir -p /content/SweetGF
else:
    print("SweetGF directory already exists")

%cd /content/SweetGF

## 3. Install Python Dependencies

In [ ]:
# Install Python packages
!pip install -q flask flask-cors requests python-dotenv pyngrok

## 4. Install Ollama

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama in background
import subprocess
import time

# Start Ollama server
ollama_process = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
print("Starting Ollama server...")
time.sleep(5)
print("Ollama server started!")

## 5. Pull Ollama Model

**Note:** This will download a large model. Choose a smaller model for faster setup.

In [ ]:
# Pull a chat model (choose one based on your needs)
# Options:
# - llama2:7b (4.1GB) - Good balance
# - mistral:7b (4.1GB) - Fast and coherent
# - gemma:7b (5.0GB) - Google's model
# - phi3:mini (2.3GB) - Smallest, fastest

MODEL = "phi3:mini"  # Change this to your preferred model

print(f"Pulling {MODEL}... This may take a few minutes.")
!ollama pull {MODEL}
print(f"{MODEL} ready!")

## 6. Install ComfyUI

In [ ]:
# Clone ComfyUI
if not os.path.exists('/content/ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
    %cd /content/ComfyUI
    !pip install -q -r requirements.txt
    !pip install -q xformers
else:
    print("ComfyUI already installed")
    %cd /content/ComfyUI

## 7. Download ComfyUI Models

**Note:** This downloads the checkpoint model specified in your config. Modify as needed.

In [ ]:
import os

# Create model directories
!mkdir -p /content/ComfyUI/models/checkpoints
!mkdir -p /content/ComfyUI/models/loras
!mkdir -p /content/ComfyUI/models/controlnet
!mkdir -p /content/ComfyUI/models/adetailer

# Download checkpoint model
# You'll need to provide URLs for your models
# Example for a popular anime model:
print("Downloading checkpoint model...")
# !wget -O /content/ComfyUI/models/checkpoints/waiNSFWIllustrious_v110.safetensors "YOUR_MODEL_URL_HERE"

# Download LoRA
# !wget -O /content/ComfyUI/models/loras/Better_detailed_pussy_and_anus_v3.0.safetensors "YOUR_LORA_URL_HERE"

# Download ControlNet
# !wget -O /content/ComfyUI/models/controlnet/control_v11p_sd15_openpose_fp16.safetensors "YOUR_CONTROLNET_URL_HERE"

print("\n⚠️ IMPORTANT: You need to manually download and place your models in the directories above.")
print("Or uncomment and add URLs to the wget commands.")
print("\nModel directories:")
print("- Checkpoints: /content/ComfyUI/models/checkpoints")
print("- LoRAs: /content/ComfyUI/models/loras")
print("- ControlNet: /content/ComfyUI/models/controlnet")

## 8. Start ComfyUI Server

In [ ]:
import subprocess
import time

%cd /content/ComfyUI

# Start ComfyUI in background
comfyui_process = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--port', '8188'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print("Starting ComfyUI server...")
time.sleep(10)
print("ComfyUI server started on port 8188!")

## 9. Configure Environment Variables

In [ ]:
%cd /content/SweetGF

# Create .env file
env_content = f"""# Ollama Configuration
OLLAMA_BASE_URL=http://localhost:11434
OLLAMA_MODEL={MODEL}

# ComfyUI Configuration
COMFYUI_BASE_URL=http://127.0.0.1:8188
COMFYUI_PATH=/content/ComfyUI

# App Configuration
FLASK_PORT=5000
FLASK_DEBUG=False

# Startup Optimization
PRIME_COMFYUI_DEEP=false

# FaceDetailer Configuration
ENABLE_FACE_DETAILER=true

# LoRA Configuration
ENABLE_LORA=true

# ControlNet Configuration
CONTROLNET_MODEL=control_v11p_sd15_openpose_fp16.safetensors
CONTROLNET_STRENGTH=0.8
"""

with open('.env', 'w') as f:
    f.write(env_content)

print("Environment configured!")
print(f"Using model: {MODEL}")

## 10. Setup Ngrok for Public Access

**Note:** You'll need an ngrok account and auth token. Get one free at https://ngrok.com

In [ ]:
from pyngrok import ngrok, conf
import getpass

# Set your ngrok auth token
print("Get your free ngrok auth token from: https://dashboard.ngrok.com/get-started/your-authtoken")
NGROK_TOKEN = getpass.getpass("Enter your ngrok auth token: ")

# Configure ngrok
conf.get_default().auth_token = NGROK_TOKEN

print("Ngrok configured!")

## 11. Start SweetGF Application

In [ ]:
import threading
from pyngrok import ngrok
import time

%cd /content/SweetGF

# Start Flask app in a thread
def run_flask():
    os.system('python app.py')

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()

print("Starting Flask app...")
time.sleep(5)

# Create ngrok tunnel
public_url = ngrok.connect(5000, bind_tls=True)

print("\n" + "="*60)
print("🌸 SweetGF is now running!")
print("="*60)
print(f"\n🌐 Access your app at: {public_url}")
print("\n⚠️ Keep this notebook running to maintain the connection.")
print("\n💡 Tips:")
print("   - The first message may take longer as the model loads")
print("   - Image generation requires models to be downloaded (see step 7)")
print("   - Share the URL with anyone you want to access the app")
print("\n" + "="*60)

## 12. Monitor Logs (Optional)

In [ ]:
# Check if services are running
import requests

def check_service(url, name):
    try:
        response = requests.get(url, timeout=5)
        print(f"✅ {name}: Running (Status {response.status_code})")
    except Exception as e:
        print(f"❌ {name}: Not responding - {str(e)}")

print("Checking services...\n")
check_service("http://localhost:11434", "Ollama")
check_service("http://localhost:8188", "ComfyUI")
check_service("http://localhost:5000/api/health", "SweetGF")

## 13. Keep Alive (Run this to prevent disconnection)

In [ ]:
# Keep the notebook alive
import time

print("Keeping session alive... Press stop button to end.")
print("Your app is accessible at the ngrok URL above.\n")

try:
    while True:
        time.sleep(60)
        print(".", end="", flush=True)
except KeyboardInterrupt:
    print("\nStopping...")

## Troubleshooting

### If Ollama isn't responding:
```python
!pkill ollama
!ollama serve &
!sleep 5
!ollama pull phi3:mini
```

### If ComfyUI isn't responding:
```python
%cd /content/ComfyUI
!pkill -f "python main.py"
!python main.py --listen 0.0.0.0 --port 8188 &
```

### If Flask isn't responding:
```python
%cd /content/SweetGF
!pkill -f "python app.py"
!python app.py &
```

### Check GPU availability:
```python
!nvidia-smi
```